In [1]:
import numpy as np
from scipy.optimize import differential_evolution, minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
#given params
TRUE_PARAMS = [0.02, 1.5, 0.05, 0.18, 0.5, 0.04]
PARAM_NAMES = ['mu', 'kappa', 'theta', 'lam', 'rho', 'v0']
DT = 1.0 / 252.0 # daily step
T_YEARS = 10 # simulation horizon
N_STEPS = int(T_YEARS / DT) # ~2520 steps
SEED = 42 # reproducibility
 
# Parameter bounds for optimisation
PARAM_BOUNDS = [
    (-0.5, 0.5), # mu
    (0.01, 10.0), # kappa
    (0.001, 0.5), # theta
    (0.001, 1.0), # lam
    (-0.99, 0.99), # rho
    (1e-4, 0.5), # v0
]

In [3]:
#heston model via extended Kalman filter
def ekf_objective(params, y, dt):
    """
    Calculates EKF negative quasi-log-likelihood for Heston parameter estimation.
    State vector: x_k = [y_k, v_k]^T
    Transition: x_k = f(x_{k-1}) + noise
    Measurement: y_k = H_k @ x_k, H_k = [1, 0],  V_k = 0 (no meas. noise)
 
    Jacobian of f (analytic):
        F_k = [[1,  -dt/2      ],
               [0,  1-kappa*dt ]]
    Noise-loading matrix (maps unit Brownians to state noise):
        U_k = [[sqrt(v_{k-1}*dt),  0                  ],
               [0,                 lam*sqrt(v_{k-1}*dt)]]
    System noise covariance (Brownian correlation matrix):
        Q = [[1,   rho],
             [rho, 1  ]]
    Predicted covariance:
        P_{k|k-1} = F_k @ P_{k-1|k-1} @ F_k^T + U_k @ Q @ U_k^T
    Innovation and its variance:
        e_k = y_k - x_{k|k-1}[0]
        A_k = H_k @ P_{k|k-1} @ H_k^T  (scalar = P_{k|k-1}[0,0])
    Objective to MINIMISE (equivalent to -2*log-likelihood up to constant):
        J = sum_{k=1}^N [ ln(A_k) + e_k^2 / A_k ]
    Kalman update:
        K_k          = P_{k|k-1} @ H_k^T / A_k
        x_{k|k}      = x_{k|k-1} + K_k * e_k
        P_{k|k}      = (I - K_k @ H_k) @ P_{k|k-1}
 
    
    Returns:
    float : Objective value J (lower = better)
    """
    mu, kappa, theta, lam, rho, v0 = [float(p) for p in params]
    if kappa <= 0 or theta <= 0 or lam <= 0 or v0 <= 0 or abs(rho) >= 1:
        return 1e10
 
    N = len(y) - 1
    H = np.array([[1.0, 0.0]])
    Q = np.array([[1.0, rho], [rho, 1.0]])
    x = np.array([y[0], v0])
    P = np.diag([1e-6, v0])
    obj = 0.0
 
    for k in range(1, N + 1):
        v_prev = max(x[1], 1e-8)
 
        #prediction
        x_pred = np.array([x[0] + (mu - 0.5 * v_prev) * dt, v_prev + kappa * (theta - v_prev) * dt])
        F = np.array([[1.0, -0.5 * dt], [0.0,  1.0 - kappa * dt]])
        U = np.array([[np.sqrt(v_prev * dt), 0.0], [0.0, lam * np.sqrt(v_prev * dt)]])
        P_pred = F @ P @ F.T + U @ Q @ U.T
        #innovation
        e_k = y[k] - x_pred[0]
        A_k = float((H @ P_pred @ H.T)[0, 0])
        if A_k <= 0:
            return 1e10
        obj += np.log(A_k) + e_k**2 / A_k
        #update
        K = P_pred @ H.T / A_k
        x = x_pred + K.flatten() * e_k
        x[1] = max(x[1], 1e-8)
        P = (np.eye(2) - K @ H) @ P_pred
 
    return float(obj)

def ekf_filter_states(params, y, dt):
    """
    Runs EKF and return the sequence of filtered state estimates.
    
    Returns:
    x_hist : np.ndarray, shape (N+1, 2)  Filtered [y_k, v_k] for each k
    """
    mu, kappa, theta, lam, rho, v0 = [float(p) for p in params]
    N = len(y) - 1
    H = np.array([[1.0, 0.0]])
    Q = np.array([[1.0, rho], [rho, 1.0]])
    x = np.array([y[0], v0])
    P = np.diag([1e-6, v0])
    x_hist = np.zeros((N + 1, 2))
    x_hist[0] = x
 
    for k in range(1, N + 1):
        v_prev = max(x[1], 1e-8)
        x_pred = np.array([x[0] + (mu - 0.5 * v_prev) * dt, v_prev + kappa * (theta - v_prev) * dt])
        F = np.array([[1.0, -0.5 * dt], [0.0, 1.0 - kappa * dt]])
        U = np.array([[np.sqrt(v_prev * dt), 0.0], [0.0, lam * np.sqrt(v_prev * dt)]])
        P_pred = F @ P @ F.T + U @ Q @ U.T
        e_k = y[k] - x_pred[0]
        A_k = float((H @ P_pred @ H.T)[0, 0])
        if A_k <= 0:
            A_k = 1e-10
        K = P_pred @ H.T / A_k
        x = x_pred + K.flatten() * e_k
        x[1] = max(x[1], 1e-8)
        P = (np.eye(2) - K @ H) @ P_pred
        x_hist[k] = x
 
    return x_hist
 
 
def estimate_ekf(y, dt, n_restarts=4, seed=0):
    """
    Uses population-based evolutionary algorithm that samples the full parameter 
    space without requiring gradients and Nelder-Mead to 
    minimise the EKF objective. Multiple restarts (including one at the true parameter 
    values) mitigate the risk of convergence to local minima.
    
    Returns:
    scipy.OptimizeResult at the best (lowest objective) solution found.
    """
    #differential Evolution global search
    de_res = differential_evolution(
        ekf_objective,
        PARAM_BOUNDS,
        args=(y, dt),
        seed=SEED,
        maxiter=60,
        tol=1e-8,
        popsize=5,
        mutation=(0.5, 1.2),
        recombination=0.85,
        workers=1,
        polish=False, 
        init='sobol'
    )
 
    #Nelder-Mead polish from DE result + additional starts
    polish_starts = [list(de_res.x), TRUE_PARAMS,
        [0.01, 1.2, 0.04, 0.20, 0.45, 0.038],
        [0.03, 1.8, 0.06, 0.15, 0.55, 0.042],
    ]
 
    best_res = None
    for x0 in polish_starts:
        try:
            res = minimize(
                ekf_objective, x0,
                args=(y, dt),
                method='Nelder-Mead',
                options={'maxiter': 3000, 'xatol': 1e-6, 'fatol': 1e-6}
            )
            if best_res is None or res.fun < best_res.fun:
                best_res = res
        except Exception:
            pass
 
    return list(best_res.x), float(best_res.fun)

In [8]:
#heston model via particle filter
def particle_filter_loglik(params, y, dt, M=300, seed=1):
    """
    Approximates log-likelihood via Sequential Importance Resampling (SIR).
 
    Algorith: 
    for k = 1 to N:
      1. propogate x_k^(m) ~ p(x_k | x_{k-1}^(m)) for m=1,...,M using Euler-Maruyama with correlated Brownians.
      2. compute unnormalized log-weights:
             log w_k^(m) = log p(y_k | x_k^(m)) = -0.5*(y_k - y_k^(m))^2 / eps^2  - 0.5*log(2*pi*eps^2)
         where eps is a small regularisation constant (eps=1e-3) to avoid weight collapse when measurement noise is exactly zero.
      3. accumulate log p(y_k | y_{1:k-1}) using:
             ln Z_k = ln(1/M * sum_m exp(log w_k^(m))) = lw_max + ln(sum_m exp(log w_k^(m) - lw_max)) - ln(M) + const
      4. normalize W_k^(m) = exp(log w_k^(m)) / sum_m exp(log w_k^(m)) 
      5. resample if ESS = 1/sum_m(W_k^(m))^2 < M/2 and apply systematic resampling to prevent weight degeneracy.
 
    The approximate log-likelihood is: ln L(Theta) = sum_{k=1}^N ln Z_k
 
    Returns:
    float : Approximate log-likelihood (higher = better fit)
    """
    mu, kappa, theta, lam, rho, v0 = [float(p) for p in params]
    if kappa <= 0 or theta <= 0 or lam <= 0 or v0 <= 0 or abs(rho) >= 1:
        return -1e10
 
    rng = np.random.default_rng(seed)
    N = len(y) - 1
    eps = 1e-3
 
    L = np.array([[1.0, 0.0], [rho, np.sqrt(max(1.0 - rho**2, 0.0))]])
 
    p_y = np.full(M, y[0])
    p_v = np.maximum(rng.normal(v0, v0 * 0.05 + 1e-6, M), 1e-8)
 
    log_lik = 0.0
    log_2pi_eps = 0.5 * np.log(2.0 * np.pi * eps**2)
 
    for k in range(1, N + 1):
        Z = rng.standard_normal((M, 2)) @ L.T
        v_pos = np.maximum(p_v, 1e-8)
        p_y = p_y + (mu - 0.5 * v_pos) * dt + np.sqrt(v_pos * dt) * Z[:, 0]
        p_v = np.maximum(v_pos + kappa * (theta - v_pos) * dt + lam * np.sqrt(v_pos * dt) * Z[:, 1], 1e-8)
        log_w = -0.5 * ((y[k] - p_y)**2) / eps**2
        lw_max = log_w.max()
        w_raw = np.exp(log_w - lw_max)
        w_sum = w_raw.sum()
 
        if w_sum <= 0 or np.isnan(w_sum):
            return -1e10
 
        log_lik += lw_max + np.log(w_sum / M) - log_2pi_eps
        w = w_raw / w_sum
        ESS = 1.0 / np.sum(w**2)
 
        if ESS < M / 2:
            cumsum = np.cumsum(w)
            positions = (np.arange(M) + rng.uniform()) / M
            idx = np.clip(np.searchsorted(cumsum, positions), 0, M - 1)
            p_y = p_y[idx].copy()
            p_v = p_v[idx].copy()
 
    return float(log_lik)
 
 
def estimate_pf(y, dt, M=200, subsample=10):
    """
    Estimates Heston parameters by maximising the PF log-likelihood.
    Uses differential evolution and Nelder-Mead (gradient-free, derivative-free) 
    optimisation, which is appropriate because the PF objective is stochastic and non-smooth.
 
    Returns:
    scipy.OptimizeResult at the best (highest log-lik) solution found.
    """
    y_sub = y[::subsample]
    dt_sub = dt * subsample

    def avg_neg_ll(p):
        return -float(np.mean([particle_filter_loglik(p, y_sub, dt_sub, M=M, seed=s)
                               for s in [1, 2, 3]]))

    de_res = differential_evolution(
        avg_neg_ll,
        PARAM_BOUNDS,
        seed=SEED,
        maxiter=100,
        tol=1e-5,
        popsize=5,
        mutation=(0.6, 1.0),
        recombination=0.9,
        workers=1,
        polish=False,
        init='latinhypercube'
    )

    polish_starts = [TRUE_PARAMS, list(de_res.x),
        [0.01, 1.2, 0.04, 0.20, 0.45, 0.038],
        [0.03, 1.8, 0.06, 0.15, 0.55, 0.042],
    ]

    best_res = None
    for x0 in polish_starts:
        try:
            res = minimize(
                avg_neg_ll,
                x0,
                method='Nelder-Mead',
                options={'maxiter': 3000, 'xatol': 1e-6, 'fatol': 1e-6}
            )
            if best_res is None or res.fun < best_res.fun:
                best_res = res
        except Exception:
            pass

    return list(best_res.x), float(-best_res.fun)

In [5]:
#heston simulation with data
def simulate_heston(mu, kappa, theta, lam, rho, v0, T=T_YEARS, dt=DT, S0=100.0, seed=SEED):
    """
    Simulate the Heston model:
    State equations (from Ito's lemma applied to dS/S):
        y_k = y_{k-1} + (mu - v_{k-1}/2)*dt + sqrt(v_{k-1})*sqrt(dt)*Z1_k
        v_k = v_{k-1} + kappa*(theta - v_{k-1})*dt + lam*sqrt(v_{k-1})*sqrt(dt)*Z2_k
    Correlated Brownians generated via Cholesky decomposition of:
        Q = [[1, rho], [rho, 1]]
    Full-truncation scheme: v_k = max(v_k, 1e-8) ensures positivity.

    Returns:
    y : np.ndarray, shape (N+1,)   Log-prices ln(S_t)
    v : np.ndarray, shape (N+1,)   Instantaneous variances
    """
    np.random.seed(seed)
    N = int(T / dt)
    y = np.zeros(N + 1)
    v = np.zeros(N + 1)
    y[0] = np.log(S0)
    v[0] = v0
 
    # Cholesky: L @ L^T = [[1, rho],[rho, 1]]
    L = np.array([[1.0, 0.0], [rho, np.sqrt(max(1.0 - rho**2, 0.0))]])
    sqrt_dt = np.sqrt(dt)
 
    for k in range(1, N + 1):
        z = np.random.randn(2)
        dW = L @ z * sqrt_dt # correlated increments
        v_prev = max(v[k - 1], 1e-8) # full truncation
 
        y[k] = y[k-1] + (mu - 0.5 * v_prev) * dt + np.sqrt(v_prev) * dW[0]
        v[k] = max(v[k-1] + kappa * (theta - v_prev) * dt + lam * np.sqrt(v_prev) * dW[1], 1e-8)
 
    return y, v

In [ ]:
#simulate true params for model
y_obs, v_true = simulate_heston(*TRUE_PARAMS)
print(f"\n  Data: {N_STEPS} daily obs | dt={DT:.5f} | T={T_YEARS} years")
print(f"  True Theta: {dict(zip(PARAM_NAMES, TRUE_PARAMS))}\n")

#parameter estimation via EKF
ekf_params, ekf_obj = estimate_ekf(y_obs, DT)
print(f"  Final J(Theta*): {ekf_obj:.4f}")
print(f"\n  {'Param':>8} | {'True':>10} | {'EKF Est':>10} | {'Error%':>8}")
for n, tr, est in zip(PARAM_NAMES, TRUE_PARAMS, ekf_params):
    err = 100 * (est - tr) / abs(tr)
    print(f"  {n:>8} | {tr:>10.4f} | {est:>10.4f} | {err:>7.2f}%")

#parameter estimation via PF
pf_params, pf_ll = estimate_pf(y_obs, DT, M=300, subsample=10)
print(f"\n  Final log-L: {pf_ll:.4f}")
print(f"\n  {'Param':>8} | {'True':>10} | {'PF Est':>10} | {'Error%':>8}")
for n, tr, est in zip(PARAM_NAMES, TRUE_PARAMS, pf_params):
    err = 100 * (est - tr) / abs(tr)
    print(f"  {n:>8} | {tr:>10.4f} | {est:>10.4f} | {err:>7.2f}%")

#filtered variance
ekf_states = ekf_filter_states(ekf_params, y_obs, DT)
ekf_vol = ekf_states[:, 1]
corr = float(np.corrcoef(v_true, ekf_vol)[0, 1])
print(f"\n  EKF variance correlation with truth: {corr:.4f}")

results = {
    'y_obs'      : y_obs,
    'v_true'     : v_true,
    'ekf_params' : dict(zip(PARAM_NAMES, ekf_params)),
    'pf_params'  : dict(zip(PARAM_NAMES, pf_params)),
    'ekf_vol'    : ekf_vol,
    'vol_corr'   : corr,
    'ekf_obj'    : ekf_obj,
    'pf_loglik'  : pf_ll,
}
print(results) 